# 01. 공공 API 원천 데이터 수집

**목적:** 장애인 고용직무분류·NCS 분류체계 전체를 받아 `data/raw/api/*.csv` 로 저장한다. (인계서 4.2·4.3)

- 키는 저장소 루트 `.env` 의 `DATA_GO_KR_KEY` 에 둔다. **노트북 셀에 키를 직접 쓰지 않는다.**
- 결과 CSV 는 커밋한다 (작은 파일). 이 CSV 가 02 노트북과 파이프라인의 입력이 된다.
- 명세는 Swagger 로만 확인한 상태다. 응답 열 이름이 문서와 다르면 `docs/data-report.md` 에 적는다 (12장 #7·#8).

In [ ]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent            # notebooks/ 에서 실행한다고 가정
sys.path.insert(0, str(ROOT / "backend"))
DATA = ROOT / "data"
import pandas as pd
pd.set_option("display.max_colwidth", 80)

from dotenv import load_dotenv   # pip install python-dotenv
load_dotenv(ROOT / ".env")
from app.ingest.data_go_kr import DataGoKrClient, fetch_jobcodes, fetch_ncs_classification
client = DataGoKrClient(os.environ["DATA_GO_KR_KEY"])

## 1) 장애인 고용직무분류 (`/job_code`)

In [ ]:
jobcode = pd.DataFrame(list(fetch_jobcodes(client)))
print(jobcode.shape); jobcode.head()

In [ ]:
# 확인: jobCdNm / jobCdLevel 중 어느 쪽이 직종명인가? (Swagger 설명이 뒤바뀌어 있음, 12장 #8)
jobcode[["jobCd", "jobCdNm", "jobCdLevel"]].sample(10, random_state=0)

In [ ]:
jobcode.to_csv(DATA / "raw/api/jobcode.csv", index=False, encoding="utf-8-sig")

## 2) NCS 분류체계 (`/ncsCdInfo`)

In [ ]:
ncs = pd.DataFrame(list(fetch_ncs_classification(client)))
print(ncs.shape); ncs.head()

In [ ]:
ncs.to_csv(DATA / "raw/api/ncs_classification.csv", index=False, encoding="utf-8-sig")